In [ ]:
from myosuite.utils import gym
import imageio
from IPython.display import Video, display
import numpy as np
import os

In [ ]:
import mujoco
env = gym.make('myoHandPoseRandom-v0', normalize_act = False)

# Gymnasium wraps envs (OrderEnforcing/PassiveEnvChecker); use base env.
base_env = env.unwrapped
base_env.init_qpos[:] = np.zeros(len(base_env.init_qpos),)
mjcModel = base_env.mj_model

# print("Muscles:")
# for i in range(mjcModel.na):
#     print([i,mjcModel.actuator(i).name])

# print("\nJoints:")
# for i in range(mjcModel.njnt):
#     print([i,mjcModel.joint(i).name])


musc_fe = [mjcModel.actuator('FDP2').id,mjcModel.actuator('EDC2').id]
L_range = round(1/mjcModel.opt.timestep)
skip_frame = 50
env.reset()

frames_sim = []
for iter_n in range(3):
    print("iteration: "+str(iter_n))
    res_sim = []
    for rp in range(2): #alternate between flexor and extensor
        for s in range(L_range):
            if not(s%skip_frame):
                frame = env.unwrapped.mj_renderer.render_offscreen(
                                width=400,
                                height=400,
                                camera_id=3)
                frames_sim.append(frame)

            ctrl = np.zeros(mjcModel.na,)

            act_val = 1 # maximum muscle activation
            if rp==0:
                ctrl[musc_fe[0]] = act_val
                ctrl[musc_fe[1]] = 0
            else:
                ctrl[musc_fe[1]] = act_val
                ctrl[musc_fe[0]] = 0
            env.step(ctrl)

os.makedirs('videos', exist_ok=True)
# make a local copy
imageio.mimwrite('videos/MyoSuite.mp4', frames_sim, fps=1.0 / env.unwrapped.dt)

# show in the notebook
display(Video('videos/MyoSuite.mp4', embed=True))